In [1]:
import numpy as np, cv2, torch, tensorflow as tf, sklearn, matplotlib.pyplot as mpl
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import layers, models
from tensorflow.keras.layers import Dense, Conv2D, MaxPool2D, Flatten, Dropout
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from pathlib import Path

In [2]:
root = Path('./MEDIAPIPE_DATA/processed_combine_asl_dataset/')
exts = {'.jpg', '.jpeg', '.png'}
skip = {'j', 'z'}
flip = {'c', 'b'}

paths = [
    (f, sub.name)
    for sub in sorted(root.iterdir()) if sub.is_dir() and sub.name.lower() not in skip
    for f in sorted(sub.iterdir()) if f.suffix.lower() in exts
]

train_paths, test_paths = train_test_split(paths, test_size=0.3, stratify=[l for _, l in paths])

def load(path_label):
    X = np.empty((len(path_label), 128, 128, 3), dtype=np.uint8)
    y = np.empty(len(path_label), dtype=object)
    n = 0
    for f, label in path_label:
        img = cv2.imread(str(f))
        if img is None:
            continue
        img = cv2.resize(img, (128, 128))
        if label.lower() in flip:
            img = cv2.flip(img, 1)
        X[n], y[n] = img, label
        n += 1
    return X[:n], y[:n].astype(str)

X_train, y_train = load(train_paths)
X_test, y_test = load(test_paths)

In [3]:
y_test = np.array([(ord(l.lower()) - ord('a')) for l in y_test])
y_train = np.array([(ord(l.lower()) - ord('a')) for l in y_train])
y_test_cat = to_categorical(y_test, 26)
y_train_cat = to_categorical(y_train, 26)

In [5]:
data_augmentation = models.Sequential([
    layers.RandomZoom(0.3, 0.3),
])
model = models.Sequential([
    layers.Input(shape=(128, 128, 3)),
    layers.Rescaling(1./255),
    data_augmentation,
    
    layers.Conv2D(32, 3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPool2D(),                    

    layers.Conv2D(64, 3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPool2D(),                    

    layers.Conv2D(128, 3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPool2D(),                    

    layers.Conv2D(256, 3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPool2D(),                    

    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(26, activation='softmax'),
])

model.compile(loss='categorical_crossentropy', optimizer=tf.keras.optimizers.Adam(5e-4), metrics=['accuracy'])
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=8,
                                     restore_best_weights=True),
]

In [5]:
for i in range (5,15):
    print(i)
    model.fit(X_train, y_train_cat, epochs=1)
    predictions = model.predict(X_test)
    predictions = np.argmax(predictions, axis=1) 
    print(classification_report(y_test, predictions))
    model.save(f'./MEDIAPIPE3_Model{i}.keras')

5


933/933 [==============================] - 135s 144ms/step
              precision    recall  f1-score   support

           0       0.98      0.95      0.97      1127
           1       1.00      0.99      1.00      1011
           2       0.99      0.99      0.99       693
           3       0.98      0.99      0.98      1490
           4       0.98      0.95      0.97      1188
           5       0.99      0.99      0.99      1517
           6       0.98      0.99      0.99      1587
           7       0.99      0.99      0.99      1561
           8       1.00      0.96      0.98      1475
          10       0.97      0.99      0.98      1624
          11       1.00      0.99      1.00      1657
          12       0.92      0.96      0.94       566
          13       0.92      0.97      0.94       684
          14       0.96      0.98      0.97      1184
          15       0.97      0.98      0.97      1086
          16       0.98      0.98      0.98      1118
          17      

AbortedError: Graph execution error:

Detected at node sequential_1/conv2d_3/Relu defined at (most recent call last):
  File "C:\Users\Jliu\AppData\Local\anaconda3\envs\asl\lib\runpy.py", line 196, in _run_module_as_main

  File "C:\Users\Jliu\AppData\Local\anaconda3\envs\asl\lib\runpy.py", line 86, in _run_code

  File "C:\Users\Jliu\AppData\Local\anaconda3\envs\asl\lib\site-packages\ipykernel_launcher.py", line 18, in <module>

  File "C:\Users\Jliu\AppData\Local\anaconda3\envs\asl\lib\site-packages\traitlets\config\application.py", line 1082, in launch_instance

  File "C:\Users\Jliu\AppData\Local\anaconda3\envs\asl\lib\site-packages\ipykernel\kernelapp.py", line 807, in start

  File "C:\Users\Jliu\AppData\Local\anaconda3\envs\asl\lib\site-packages\tornado\platform\asyncio.py", line 211, in start

  File "C:\Users\Jliu\AppData\Local\anaconda3\envs\asl\lib\asyncio\base_events.py", line 603, in run_forever

  File "C:\Users\Jliu\AppData\Local\anaconda3\envs\asl\lib\asyncio\base_event

In [4]:
model = tf.keras.models.load_model('./MEDIAPIPE3_Model5.keras')